# 04 · Aggregations & `GROUP BY`

Aggregate functions summarize many rows into one value:
- `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`
- `GROUP BY` computes an aggregate *per group*
- `HAVING` filters groups (like `WHERE` but after aggregation)

**Order of evaluation:** `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY`.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Aggregate over the whole table

In [ ]:
%%sql
SELECT
    COUNT(*)        AS num_products,
    AVG(unit_price) AS avg_price,
    MIN(unit_price) AS cheapest,
    MAX(unit_price) AS priciest
FROM products;

## `COUNT(*)` vs `COUNT(column)`
`COUNT(column)` ignores NULLs. We have 12 customers but not all have an email:

In [ ]:
%%sql
SELECT COUNT(*) AS total, COUNT(email) AS with_email FROM customers;

## `GROUP BY` — one row per group
How many products per category?

In [ ]:
%%sql
SELECT category_id, COUNT(*) AS num_products, ROUND(AVG(unit_price), 2) AS avg_price
FROM products
GROUP BY category_id
ORDER BY category_id;

## Counting customers per country

In [ ]:
%%sql
SELECT country, COUNT(*) AS customers
FROM customers
GROUP BY country
ORDER BY customers DESC;

## `HAVING` — filter the groups
`WHERE` filters rows *before* grouping; `HAVING` filters *after*. Show only
categories that have more than 3 products:

In [ ]:
%%sql
SELECT category_id, COUNT(*) AS num_products
FROM products
GROUP BY category_id
HAVING COUNT(*) > 3;

## `WHERE` + `GROUP BY` + `HAVING` together
Average price of products **in stock** per category, keeping only categories whose average exceeds $50:

In [ ]:
%%sql
SELECT category_id, ROUND(AVG(unit_price), 2) AS avg_price
FROM products
WHERE in_stock > 0
GROUP BY category_id
HAVING AVG(unit_price) > 50
ORDER BY avg_price DESC;

## Practice

**✏️ Exercise 1.** How many orders are there for each `status`?

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT status, COUNT(*) AS n FROM orders GROUP BY status;

**✏️ Exercise 2.** What is the total quantity sold for each `product_id` in `order_items`? Show the top 5.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT product_id, SUM(quantity) AS total_qty
FROM order_items
GROUP BY product_id
ORDER BY total_qty DESC
LIMIT 5;

**✏️ Exercise 3.** Find countries that have 2 or more customers.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT country, COUNT(*) AS customers
FROM customers
GROUP BY country
HAVING COUNT(*) >= 2;

### ✅ Recap
Aggregates collapse rows; `GROUP BY` aggregates per group; `HAVING` filters
groups. Remember `WHERE` (rows) vs `HAVING` (groups).

**Next:** `05_joins.ipynb` — combining tables.